In [2]:
#import libraries

import pandas as pd
from neo4j import GraphDatabase
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

In [3]:
#load extracted entities dataset

entity_df=pd.read_csv(
    "../Data/extracted_entities.csv"
)
print(entity_df.shape)
display(entity_df.head())

(9276, 5)


,row_id,original_text,entity_text,entity_label,confidence
0,0,\n Patient Query:\n I woke up this morni...,panadol,medication,0.6307
1,0,\n Patient Query:\n I woke up this morni...,benign paroxysmal positional vertigo,medical_condition,0.7313
2,0,\n Patient Query:\n I woke up this morni...,BPPV,medical_condition,0.5871
3,0,\n Patient Query:\n I woke up this morni...,peripheral vertigo,medical_condition,0.5706
4,0,\n Patient Query:\n I woke up this morni...,dizziness,symptom,0.8374


In [4]:
from dotenv import load_dotenv
import os
load_dotenv()

URI=os.getenv("NEO4J_URI")
USERNAME=os.getenv("NEO4J_USERNAME")
PASSWORD=os.getenv("NEO4J_PASSWORD")

In [5]:
#create neo4j driver

driver=GraphDatabase.driver(
    URI,
    auth=(USERNAME,PASSWORD)
)

print("neo4j connected successfully")

neo4j connected successfully


In [6]:
#test neo4j connection

with driver.session() as session:

    result=session.run(
        "RETURN 'Neo4j Connected' AS message"
    )

    for record in result:

        print(record["message"])

Neo4j Connected


In [7]:
#create constraints

with driver.session() as session:
    session.run("""
    CREATE CONSTRAINT symptom_name IF NOT EXISTS
    FOR (s:Symptom)
    REQUIRE s.name IS UNIQUE
    """)
    session.run("""
    CREATE CONSTRAINT disease_name IF NOT EXISTS
    FOR (d:Disease)
    REQUIRE d.name IS UNIQUE
    """)
    session.run("""
    CREATE CONSTRAINT medication_name IF NOT EXISTS
    FOR (m:Medication)
    REQUIRE m.name IS UNIQUE
    """)
print("constraints created")

constraints created


In [8]:
#separate entities

symptoms=entity_df[
    entity_df["entity_label"]=="symptom"
]

diseases=entity_df[
    entity_df["entity_label"]=="disease"
]

medications=entity_df[
    entity_df["entity_label"]=="medication"
]

medical_conditions=entity_df[
    entity_df["entity_label"]=="medical_condition"
]

print(symptoms.shape)
print(diseases.shape)
print(medications.shape)
print(medical_conditions.shape)

(1915, 5)
(1185, 5)
(1656, 5)
(1210, 5)


In [9]:
#create symptom nodes

with driver.session() as session:
    for symptom in tqdm(
        symptoms["entity_text"].unique()
    ):
        session.run("""
        MERGE (s:Symptom {
            name:$name
        })

        """,
        name=symptom.lower()
        )
print("symptom nodes created")

100%|███████████████████████████████████████████████████████████████████████████████| 645/645 [00:06<00:00, 102.76it/s]

symptom nodes created


In [10]:
#create disease nodes

with driver.session() as session:
    for disease in tqdm(
        diseases["entity_text"].unique()
    ):
        session.run("""
        MERGE (d:Disease {
            name:$name
        })
        """,
        name=disease.lower()
        )

print("disease nodes created")

100%|███████████████████████████████████████████████████████████████████████████████| 608/608 [00:03<00:00, 158.17it/s]

disease nodes created


In [11]:
#create medication nodes

with driver.session() as session:
    for medication in tqdm(
        medications["entity_text"].unique()
    ):
        session.run("""
        MERGE (m:Medication {
            name:$name
        })
        """,
        name=medication.lower()
        )

print("medication nodes created")

100%|█████████████████████████████████████████████████████████████████████████████| 1088/1088 [00:06<00:00, 169.25it/s]

medication nodes created


In [12]:
#create medical condition nodes

with driver.session() as session:
    for condition in tqdm(
        medical_conditions["entity_text"].unique()
    ):
        session.run("""
        MERGE (c:MedicalCondition {
            name:$name
        })
        """,
        name=condition.lower()
        )
print("medical condition nodes created")

100%|███████████████████████████████████████████████████████████████████████████████| 659/659 [00:05<00:00, 129.29it/s]

medical condition nodes created


In [14]:
#create symptom to disease relationships

with driver.session() as session:
    for index,row in tqdm(
        entity_df.iterrows(),
        total=len(entity_df)
    ):
        try:
            row_id=row["row_id"]
            current_entity=row["entity_text"].lower()
            current_label=row["entity_label"]
            same_rows=entity_df[
                entity_df["row_id"]==row_id
            ]
            symptoms_in_row=same_rows[
                same_rows["entity_label"]=="symptom"
            ]
            diseases_in_row=same_rows[
                same_rows["entity_label"]=="disease"
            ]
            conditions_in_row=same_rows[
                same_rows["entity_label"]=="medical_condition"
            ]
            for _,symptom_row in symptoms_in_row.iterrows():
                symptom_name=symptom_row[
                    "entity_text"
                ].lower()
                for _,disease_row in diseases_in_row.iterrows():
                    disease_name=disease_row[
                        "entity_text"
                    ].lower()
                    session.run("""
                    MATCH (s:Symptom {
                        name:$symptom
                    })
                    MATCH (d:Disease {
                        name:$disease
                    })
                    MERGE (s)-[:RELATED_TO]->(d)
                    """,
                    symptom=symptom_name,
                    disease=disease_name
                    )
                for _,condition_row in conditions_in_row.iterrows():
                    condition_name=condition_row[
                        "entity_text"
                    ].lower()
                    session.run("""
                    MATCH (s:Symptom {
                        name:$symptom
                    })
                    MATCH (c:MedicalCondition {
                        name:$condition
                    })
                    MERGE (s)-[:RELATED_TO]->(c)
                    """,
                    symptom=symptom_name,
                    condition=condition_name
                    )
        except Exception as e:
            print(e)
print("relationships created")

100%|██████████████████████████████████████████████████████████████████████████████| 9276/9276 [03:14<00:00, 47.74it/s]

relationships created


In [15]:
#create medication relationships

with driver.session() as session:
    for index,row in tqdm(
        entity_df.iterrows(),
        total=len(entity_df)
    ):
        try:
            row_id=row["row_id"]
            same_rows=entity_df[
                entity_df["row_id"]==row_id
            ]
            conditions=same_rows[
                same_rows["entity_label"]=="medical_condition"
            ]
            medications=same_rows[
                same_rows["entity_label"]=="medication"
            ]
            for _,condition_row in conditions.iterrows():
                condition_name=condition_row[
                    "entity_text"
                ].lower()
                for _,medication_row in medications.iterrows():
                    medication_name=medication_row[
                        "entity_text"
                    ].lower()
                    session.run("""
                    MATCH (c:MedicalCondition {
                        name:$condition
                    })
                    MATCH (m:Medication {
                        name:$medication
                    })
                    MERGE (c)-[:TREATED_BY]->(m)
                    """,

                    condition=condition_name,
                    medication=medication_name
                    )
        except Exception as e:
            print(e)

print("medication relationships created")

100%|█████████████████████████████████████████████████████████████████████████████| 9276/9276 [01:31<00:00, 101.61it/s]

medication relationships created


In [16]:
#count graph nodes

with driver.session() as session:
    result=session.run("""
    MATCH (n)
    RETURN count(n) AS total_nodes

    """)

    for record in result:
        print(
            "Total Nodes:",
            record["total_nodes"]
        )

Total Nodes: 2728


In [17]:
#count graph relationships

with driver.session() as session:
    result=session.run("""
    MATCH ()-[r]->()
    RETURN count(r) AS total_relationships
    """)
    for record in result:
        print(
            "Total Relationships:",
            record["total_relationships"]
        )

Total Relationships: 3516


In [18]:
#sample graph query

with driver.session() as session:
    result=session.run("""
    MATCH (s:Symptom)-[:RELATED_TO]->(d)
    RETURN s.name AS symptom,
           d.name AS disease
    LIMIT 10
    """)
    for record in result:
        print(
            record["symptom"],
            "->",
            record["disease"]
        )

dizziness -> osteoporosis
dizziness -> cancer
dizziness -> cervical spondylosis
dizziness -> breast cancers
dizziness -> isthmic spondylolisthesis
dizziness -> islamic spondylolisthesis
dizziness -> asthma
dizziness -> cardiac disease
dizziness -> diabetic
dizziness -> kidney infection


In [19]:
#close neo4j connection

driver.close()

print("neo4j connection closed")

neo4j connection closed
